In [6]:
# Gujarati_Light.ipynb
# Purpose: Train a CPU-friendly "Light" Gujarati fake-news classifier (TF-IDF + LinearSVC).
# Saves model and metrics to: models/light/gujarati_light_model/
# Usage: Paste entire file into one notebook cell (or run as script). Adjust FILE_PATH or COL names if needed.

# ---------------------------
FILE_PATH = r"S:\MCA PRACTICAL\3rd sem\Minor Project\TruthLens\Data\Preprocessed\Preprocessed_Gujrati_data.csv"
TEXT_COL  = "clean_joined"   # change if your CSV uses different cleaned-text column (e.g., "clean")
LABEL_COL = "label"
RANDOM_STATE = 42
TEST_SIZE = 0.40
VAL_SIZE = 0.20   # fraction of train used for validation
OUT_DIR = r"models\light\gujarati_light_model"
MIN_TEXT_LEN = 15   # smaller languages may have shorter lines; adjust if needed
FAST_TRAIN_MAX = 4000  # subsample train set to at most this many rows for fast runs
# ---------------------------


In [7]:
import os, json
os.makedirs(OUT_DIR, exist_ok=True)

print("Step 1/9 — Loading data from CSV...")
import pandas as pd
from pathlib import Path
p = Path(FILE_PATH)
if not p.exists():
    raise FileNotFoundError(f"Data file not found: {FILE_PATH}")

def load_data(path, usecols):
    # Try simple read, else fallback to robust chunked read (handles messy CSVs)
    try:
        return pd.read_csv(path, usecols=usecols, encoding="utf-8", on_bad_lines="skip", low_memory=True)
    except Exception:
        import csv as _csv, sys
        try:
            _csv.field_size_limit(sys.maxsize)
        except Exception:
            pass
        chunks = []
        for chunk in pd.read_csv(path, usecols=usecols, engine="python", encoding="utf-8",
                                 on_bad_lines="skip", chunksize=50000, sep=",",
                                 quotechar=None, quoting=_csv.QUOTE_NONE, escapechar="\\"):
            chunks.append(chunk)
        return pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=usecols)

df = load_data(FILE_PATH, usecols=[TEXT_COL, LABEL_COL])
print(f"Raw rows loaded: {len(df)}")

# ---- Basic cleaning and filtering ----
print("Step 2/9 — Cleaning and basic filtering...")
df = df.dropna(subset=[TEXT_COL, LABEL_COL]).copy()
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].str.len() >= MIN_TEXT_LEN].reset_index(drop=True)
before = len(df)
df = df.drop_duplicates(subset=[TEXT_COL]).reset_index(drop=True)
print(f"Removed duplicates: {before - len(df)}  (remaining rows: {len(df)})")

# ---- Label normalization ----
print("Step 3/9 — Normalizing labels...")
def normalize_labels(ser):
    s = ser.copy()
    if pd.api.types.is_integer_dtype(s) or pd.api.types.is_float_dtype(s):
        return s.fillna(0).astype(int)
    s = s.astype(str).str.strip().str.lower()
    # include some common gujarati/hindi tokens if present; adapt if your labels use words
    mapping = {"fake":1,"false":1,"ફેક્સ":1,"ફેક":1,"jb_fake":1,"1":1,
               "real":0,"true":0,"સાચો":0,"genuine":0,"0":0}
    mapped = s.map(mapping)
    if mapped.isnull().any():
        # fallback heuristic: if string contains words meaning 'true/real' mark 0 else 1
        mapped = mapped.fillna(s.apply(lambda x: 0 if any(w in x for w in ["real","true","સાચો","correct"]) else 1))
    return mapped.astype(int)

y = normalize_labels(df[LABEL_COL])
X = df[TEXT_COL].astype(str)
print("Label distribution (after normalization):", y.value_counts(dropna=False).to_dict())

# ---- Train / val / test split (stratified) ----
print("Step 4/9 — Creating train/validation/test splits (stratified)...")
from sklearn.model_selection import train_test_split, StratifiedShuffleSplit
X_train_all, X_test, y_train_all, y_test = train_test_split(X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)
sss = StratifiedShuffleSplit(n_splits=1, test_size=VAL_SIZE, random_state=RANDOM_STATE)
tr_idx, val_idx = next(sss.split(X_train_all, y_train_all))
X_tr, y_tr = X_train_all.iloc[tr_idx].reset_index(drop=True), y_train_all.iloc[tr_idx].reset_index(drop=True)
X_val, y_val = X_train_all.iloc[val_idx].reset_index(drop=True), y_train_all.iloc[val_idx].reset_index(drop=True)
print(f"Train: {len(X_tr)}, Val: {len(X_val)}, Test: {len(X_test)}")

# ---- Build TF-IDF + LinearSVC pipeline (define named cleaner to avoid pickling lambda issues) ----
print("Step 5/9 — Building TF-IDF + LinearSVC pipeline...")
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.svm import LinearSVC

import re
def gu_clean_texts(X_list):
    # Named function (picklable) to clean list/iterator of texts
    url_pat = re.compile(r"http\S+|www\.\S+")
    handle_pat = re.compile(r"[@#]\w+")
    multi_ws = re.compile(r"\s+")
    out = []
    for s in X_list:
        s = str(s).lower()
        s = url_pat.sub(" ", s)
        s = handle_pat.sub(" ", s)
        s = re.sub(r"[^\w\s]", " ", s, flags=re.UNICODE)   # remove punctuation
        s = multi_ws.sub(" ", s).strip()
        out.append(s)
    return out

# Vectorizers
word = TfidfVectorizer(analyzer='word', token_pattern=r'(?u)\b\w+\b',
                       ngram_range=(1,2), min_df=2, max_df=0.95,
                       sublinear_tf=True, strip_accents='unicode', stop_words=None)
# stop_words=None because Gujarati stoplist not provided; TF-IDF still works.
char = TfidfVectorizer(analyzer='char', ngram_range=(3,5), min_df=2, sublinear_tf=True)

# Build a simple pipeline manually: apply cleaning -> transform texts -> selection -> classifier
from sklearn.base import TransformerMixin, BaseEstimator
class CleanerTransformer(BaseEstimator, TransformerMixin):
    def __init__(self, func):
        self.func = func
    def fit(self, X, y=None):
        return self
    def transform(self, X):
        return self.func(X)

cleaner = CleanerTransformer(gu_clean_texts)
feats = FeatureUnion([('word', word), ('char', char)])
select = SelectKBest(chi2, k=10000)   # keep smaller K for light model
clf = LinearSVC(random_state=RANDOM_STATE, C=1.0, max_iter=20000)

pipe = Pipeline([('clean', cleaner), ('feats', feats), ('select', select), ('clf', clf)])

# ---- Training: choose fast mode (subsample) for light model ----
print("Step 6/9 — Training model (fast mode: subsample training data for speed)...")
from sklearn.model_selection import StratifiedKFold, GridSearchCV
skf = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)

# Small grid (or single config) — keep it quick. Using single-config here is fastest and stable.
param_grid = {
    'feats__word__ngram_range': [(1,1)],
    'feats__word__min_df': [2],
    'feats__char__ngram_range': [(3,5)],
    'select__k': [10000],
    'clf__C': [1.0]
}

# Subsample train set proportionally by class to FAST_TRAIN_MAX
if len(X_tr) > FAST_TRAIN_MAX:
    print(f"Fast training: subsample training data from {len(X_tr)} -> {FAST_TRAIN_MAX}")
    df_tr = pd.DataFrame({ 'text': X_tr, 'label': y_tr })
    n = FAST_TRAIN_MAX
    # stratified proportional sample
    df_sub = df_tr.groupby('label', group_keys=False).apply(lambda g: g.sample(max(1, int(n * len(g)/len(df_tr))), random_state=RANDOM_STATE)).reset_index(drop=True)
    X_tr_sub = df_sub['text'].tolist()
    y_tr_sub = df_sub['label'].astype(int).tolist()
else:
    X_tr_sub = X_tr.tolist()
    y_tr_sub = y_tr.tolist()

# Use GridSearchCV with n_jobs=1 to avoid orphan processes on student machines
grid = GridSearchCV(pipe, param_grid, scoring='f1', cv=skf, verbose=1, n_jobs=1)
grid.fit(X_tr_sub, y_tr_sub)
print("Best params:", grid.best_params_)
print("Best CV f1: {:.4f}".format(grid.best_score_))

best_pipeline = grid.best_estimator_

# ---- Calibrate classifier to output probabilities ----
print("Step 7/9 — Calibrating classifier to obtain probabilities...")
from sklearn.calibration import CalibratedClassifierCV
calibrated = CalibratedClassifierCV(best_pipeline, cv=2, method='sigmoid')  # cv=2 for speed
calibrated.fit(X_tr_sub, y_tr_sub)
print("Calibration done.")

# ---- Tune decision threshold on validation set ----
print("Step 8/9 — Tuning decision threshold on validation set (maximize F1)...")
import numpy as np
from sklearn.metrics import f1_score, accuracy_score, roc_auc_score, classification_report, confusion_matrix

val_prob = calibrated.predict_proba(X_val.tolist())[:, 1]
best_th, best_f1 = 0.5, -1.0
for t in np.linspace(0.2, 0.8, 61):
    f1 = f1_score(y_val, (val_prob >= t).astype(int))
    if f1 > best_f1:
        best_f1, best_th = f1, t
print(f"Chosen threshold on validation: {best_th:.3f}  (val F1={best_f1:.4f})")

# ---- Evaluate on test set ----
print("Step 9/9 — Evaluating on test set and saving artifacts...")
test_prob = calibrated.predict_proba(X_test.tolist())[:, 1]
y_pred = (test_prob >= best_th).astype(int)

metrics = {
    "test_accuracy": float(accuracy_score(y_test, y_pred)),
    "test_f1": float(f1_score(y_test, y_pred)),
    "test_roc_auc": None,
    "threshold": float(best_th),
    "n_train_total": int(len(X_tr)),
    "n_train_used": int(len(X_tr_sub)),
    "n_val": int(len(X_val)),
    "n_test": int(len(X_test)),
}

try:
    metrics["test_roc_auc"] = float(roc_auc_score(y_test, test_prob))
except Exception:
    metrics["test_roc_auc"] = None

print("Test Accuracy:", metrics["test_accuracy"])
print("Test F1:", metrics["test_f1"])
print("Test ROC-AUC:", metrics["test_roc_auc"])
print("Classification report:\n", classification_report(y_test, y_pred, digits=4))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred))

# ---- Save model + metrics + info ----
print("Saving model and metrics to disk...")
from joblib import dump
model_path = os.path.join(OUT_DIR, "model.joblib")
# Save calibrated pipeline + threshold
dump({"pipeline": calibrated, "threshold": float(best_th)}, model_path)
print(f"Saved model: {model_path}")

metrics_path = os.path.join(OUT_DIR, "metrics.json")
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump(metrics, f, indent=2)
print(f"Saved metrics: {metrics_path}")

info = {
    "purpose": "Light Gujarati fake-news classifier for frontend Light tier",
    "file_path_used": FILE_PATH,
    "text_col": TEXT_COL,
    "label_col": LABEL_COL,
    "random_state": RANDOM_STATE,
    "model_type": "tfidf_word+char + LinearSVC (calibrated)",
    "training_date": pd.Timestamp.now().isoformat()
}
info_path = os.path.join(OUT_DIR, "info.json")
with open(info_path, "w", encoding="utf-8") as f:
    json.dump(info, f, indent=2)
print(f"Saved info: {info_path}")

print("All done. Light Gujarati model ready.")
print("Usage (later in backend): obj = joblib.load(model.joblib); probs = obj['pipeline'].predict_proba([text]); label = (probs[:,1] >= obj['threshold']).astype(int)")


Step 1/9 — Loading data from CSV...
Raw rows loaded: 17659
Step 2/9 — Cleaning and basic filtering...
Removed duplicates: 2659  (remaining rows: 14781)
Step 3/9 — Normalizing labels...
Label distribution (after normalization): {0: 8639, 1: 6142}
Step 4/9 — Creating train/validation/test splits (stratified)...
Train: 7094, Val: 1774, Test: 5913
Step 5/9 — Building TF-IDF + LinearSVC pipeline...
Step 6/9 — Training model (fast mode: subsample training data for speed)...
Fast training: subsample training data from 7094 -> 4000
Fitting 3 folds for each of 1 candidates, totalling 3 fits


C:\Users\sanyam\AppData\Local\Temp\ipykernel_10156\141602687.py:137: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  df_sub = df_tr.groupby('label', group_keys=False).apply(lambda g: g.sample(max(1, int(n * len(g)/len(df_tr))), random_state=RANDOM_STATE)).reset_index(drop=True)


Best params: {'clf__C': 1.0, 'feats__char__ngram_range': (3, 5), 'feats__word__min_df': 2, 'feats__word__ngram_range': (1, 1), 'select__k': 10000}
Best CV f1: 0.9737
Step 7/9 — Calibrating classifier to obtain probabilities...
Calibration done.
Step 8/9 — Tuning decision threshold on validation set (maximize F1)...
Chosen threshold on validation: 0.640  (val F1=0.9782)
Step 9/9 — Evaluating on test set and saving artifacts...
Test Accuracy: 0.9785219008963301
Test F1: 0.9739273249846028
Test ROC-AUC: 0.9973700425089314
Classification report:
               precision    recall  f1-score   support

           0     0.9757    0.9878    0.9817      3456
           1     0.9826    0.9654    0.9739      2457

    accuracy                         0.9785      5913
   macro avg     0.9792    0.9766    0.9778      5913
weighted avg     0.9786    0.9785    0.9785      5913

Confusion matrix:
 [[3414   42]
 [  85 2372]]
Saving model and metrics to disk...
Saved model: models\light\gujarati_light_m